In [1]:
from unsloth import FastLanguageModel

def load_model(model_name: str, max_sequence_length: int = 2048, load_in_4bit: bool = True):

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_sequence_length,
        load_in_4bit = load_in_4bit,  #
        # token = "YOUR_HF_TOKEN", # HF Token for gated models
    )
    return model, tokenizer


model, tokenizer = load_model("unsloth/mistral-7b-instruct-v0.3-bnb-4bit")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/anaconda/envs/finetune_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-16GB. Num GPUs = 1. Max memory: 15.773 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [2]:
import pandas as pd

# load train set
train_data = pd.read_json("datasets/alpaca_train.jsonl", orient="records", lines=True)


from datasets import Dataset

# df is your pandas DataFrame
hf_dataset = Dataset.from_pandas(train_data)

chat_template = """Beantwoord de volgende vraag zo goed mogelijk.

### Vraag:
{INPUT}

### Antwoord:
{OUTPUT}
"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def formatting_prompts_func(data):
    inputs       = data["input"]
    outputs      = data["output"]
    texts = []
    for input, output in zip(inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = chat_template.format(INPUT=input, OUTPUT=output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }


formatted_train_data = hf_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=hf_dataset.column_names,)

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 41370/41370 [00:00<00:00, 221257.83 examples/s]


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
import wandb

WANDB_PROJECT = "dutch-mistral"
WANDB_RUN_NAME = "TEST"

run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME, 
    config=model.peft_config
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/azureuser/.netrc.
wandb: Currently logged in as: silkeplessers (silkeplessers-microsoft) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


In [ ]:
from trl import SFTConfig, SFTTrainer

RUN_NR = 'test'

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_train_data,
    dataset_text_field = "text",
    max_seq_length = 2048, # Must set this to your model's max context length
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,  #effective batch size = per_device_train_batch_size * gradient_accumulation_steps
        warmup_steps = 300,
        #num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 10,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = f"outputs/run{RUN_NR}",
        report_to = "wandb", # Use TrackIO/WandB etc
        disable_tqdm = True, # Enable this to disable tqdm progress bars
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=10): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 41370/41370 [00:03<00:00, 11394.10 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [11]:
# @title Show current memory stats
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla V100-PCIE-16GB. Max memory = 15.773 GB.
6.766 GB of memory reserved.


In [6]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 41,370 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!
{'loss': 1.6256, 'grad_norm': 3.7686638832092285, 'learning_rate': 0.0, 'epoch': 0.00038675368624607205}
{'loss': 1.6031, 'grad_norm': 3.5171923637390137, 'learning_rate': 6.666666666666667e-07, 'epoch': 0.0007735073724921441}
{'loss': 1.7041, 'grad_norm': 3.6795308589935303, 'learning_rate': 1.3333333333333334e-06, 'epoch': 0.001160261058738216}
{'loss': 1.6692, 'grad_norm': 3.892085552215576, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.0015470147449842882}
{'loss': 1.6715, 'grad_norm': 3.5304453372955322, 'learning_rate': 2.666666666666667e-06, 'epoch': 0.0019337684312303602}
{'loss': 1.6633, 'grad_norm': 2.55360746383667, 'learning_rate': 3.3333333333333333e-06, 'epoch': 0.002320522117476432}
{'loss': 1.6566, 'grad_norm': 2.404595136642456, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.0027072758037225042}
{'loss': 1.6566, 'grad_norm': 1.7556253671646118, 'learning_rate': 4.666666666666667e-06, 'epoch': 0.00309402948

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train/epoch,▁▂▃▃▄▅▆▆▇██
train/global_step,▁▂▃▃▄▅▆▆▇██
train/grad_norm,█▇▇█▇▄▃▁▃▂
train/learning_rate,▁▂▃▃▄▅▆▆▇█
train/loss,▃▂▄▄▄▄▃▃█▁
total_flos,1265665924571136.0
train/epoch,0.00387
train/global_step,10
train/grad_norm,1.96848
train/learning_rate,1e-05
train/loss,1.5486


In [9]:
model.save_pretrained(f"outputs/run{RUN_NR}/mistral_lora_run{RUN_NR}")  # Local saving
tokenizer.save_pretrained(f"outputs/run{RUN_NR}/mistral_lora_run{RUN_NR}")

('outputs/runtest/mistral_lora_runtest/tokenizer_config.json',
 'outputs/runtest/mistral_lora_runtest/special_tokens_map.json',
 'outputs/runtest/mistral_lora_runtest/chat_template.jinja',
 'outputs/runtest/mistral_lora_runtest/tokenizer.model',
 'outputs/runtest/mistral_lora_runtest/added_tokens.json',
 'outputs/runtest/mistral_lora_runtest/tokenizer.json')

In [ ]:
artifact = wandb.Artifact(f"mistral-qlora-run{RUN_NR}", type="model")
artifact.add_dir(f"outputs/run{RUN_NR}/mistral_lora_run{RUN_NR}")

run.log_artifact(artifact)   # use the run from cell 3
run.finish()                 # finish once, at the end


wandb: Adding directory to artifact (outputs/runtest/mistral_lora_runtest)... 

Done. 1.3s


UsageError: Run (bulrbap4) is finished. The call to `log_artifact` will be ignored. Please make sure that you are using an active run.